#### **TF-IDF**

Defining the TF-IDF class

In [26]:
from src.retrievers.tf_idf import TFIDF
from src.retrievers.base_retriever import Retriever

class DevTFIDF(TFIDF):
    def __init__(self, documents):
        Retriever.__init__(self, documents)   # builds self.index: DocumentDataset

        # these 3 are used only to call GetDocumentsTFIDFVectors. 
        # everywhere else, we just use the methods directly.
        self.corpus_vocabulary = self.GetCorpusVocabulary()
        self.idf = self.GetInverseDocumentFrequencies()
        self.tfidf_matrix = self.GetDocumentsTFIDFVectors()

Instantiating a dummy dataset (of 3 dummy documents) just for testing in this notebook. 

* Note the difference between `raw_docs` and `docs` is the `DocumentDataset` class. 

* The `raw_docs` are not wrapped with the `DocumentDataset` class.

* This happens when passed to the `Retriever` class, in `__BuildIndex`, which we can call using `.index` on the TFIDF `Retriever`.

In [27]:
raw_docs = [{
    'text': ['the cat sat on the mat', 'the dog sat on the log', 'cats and dogs'],
    'document_id': ['d0', 'd1', 'd2']}]

Instantiating the TF-IDF retriever class for testing utility functions in this notebook

In [28]:
tf_idf = DevTFIDF(raw_docs) # instantiate the class (same way as in main.py), to access the methods from TFIDF retriever

To access the document objects we use the `docs` directly as this built as the `DocumentDataset` class.

In [29]:
docs = tf_idf.index # the DocumentDataset the retriever built
docs.documents[0], docs.documents[1], docs.documents[2]

(<src.data_utils.document.Document at 0x1117497c0>,
 <src.data_utils.document.Document at 0x111748110>)

To access the attributes of each `Document` object directly (text, document_id), we use `docs`

In [30]:
print(docs.documents[0].GetText())
print(docs.documents[0].GetId())

the cat sat on the mat
d0


For example, to get the vocabulary or term counts of a document:

In [31]:
print(tf_idf.GetDocumentVocabulary(docs.documents[0]))
print(tf_idf.GetDocumentTermCounts(docs.documents[0]))

{'on', 'sat', 'the', 'mat', 'cat'}
{'on': 1, 'sat': 1, 'the': 2, 'mat': 1, 'cat': 1}


To access the methods of the `TFIDF` class instance, we use the `raw_docs`.

Preprocess text (used directly on any string)

In [32]:
print(tf_idf.PreprocessText(raw_docs[0]["text"][0]))
print(tf_idf.PreprocessText(raw_docs[0]["document_id"][0]))

['the', 'cat', 'sat', 'on', 'the', 'mat']
['d0']


Compute the *Document Frequencies* and *Inverse Document Frequencies* for all `raw_docs`

In [33]:
print(tf_idf.GetDocumentFrequencies())
print(tf_idf.GetInverseDocumentFrequencies())

{'on': 2, 'sat': 2, 'the': 2, 'mat': 1, 'cat': 1, 'log': 1, 'dog': 1, 'dogs': 1, 'cats': 1, 'and': 1}
{'on': np.float64(0.4054651081081644), 'sat': np.float64(0.4054651081081644), 'the': np.float64(0.4054651081081644), 'mat': np.float64(1.0986122886681098), 'cat': np.float64(1.0986122886681098), 'log': np.float64(1.0986122886681098), 'dog': np.float64(1.0986122886681098), 'dogs': np.float64(1.0986122886681098), 'cats': np.float64(1.0986122886681098), 'and': np.float64(1.0986122886681098)}


The next thing we can do is compute the TFIDF vectors.

`GetDocumentsTFIDFVector` constructs a *sparse* matrix (zero values are not shown) which includes the TF-IDF scores. 

Specifically, it returns a n_documents x n_terms matrix whose element (i, j) is the tf-idf score of the j'th term in the i'th document.

In [44]:
for idx, word in enumerate(tf_idf.GetCorpusVocabulary()):
    print(f'{idx} : {word}')

0 : on
1 : sat
2 : the
3 : mat
4 : and
5 : log
6 : cats
7 : dogs
8 : dog
9 : cat


In [35]:
print(tf_idf.GetDocumentsTFIDFVectors())

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 13 stored elements and shape (3, 10)>
  Coords	Values
  (0, 0)	0.20273254811763763
  (0, 1)	0.20273254811763763
  (0, 2)	0.40546509623527527
  (0, 3)	0.5493061542510986
  (0, 9)	0.5493061542510986
  (1, 0)	0.20273254811763763
  (1, 1)	0.20273254811763763
  (1, 2)	0.40546509623527527
  (1, 5)	0.5493061542510986
  (1, 8)	0.5493061542510986
  (2, 4)	1.0986123085021973
  (2, 6)	1.0986123085021973
  (2, 7)	1.0986123085021973


So this gives us a 3x10 matrix (documents * corpus vocab) containing the tf-idf scores.

In [51]:
print(tf_idf.tfidf_matrix.toarray())


[[0.20273255 0.20273255 0.4054651  0.54930615 0.         0.
  0.         0.         0.         0.54930615]
 [0.20273255 0.20273255 0.4054651  0.         0.         0.54930615
  0.         0.         0.54930615 0.        ]
 [0.         0.         0.         0.         1.0986123  0.
  1.0986123  1.0986123  0.         0.        ]]


These will serve as the embeddings that we will later compare our queries to, to decide what documents are most relevant to a query.

### Queries

Now that we have the `Document` and the `Retriever` class defined, we can compute the *Term Frequencies* and *Inverse Document Frequencies* for all the documents.

But our goal is to be able to *retrieve* the most relevant *document*, based on a *query*.

So we need a `Query` class for the queries, which we can use to compute the *similarity score* to each `Document` using the `Retriever` class.

Defining the `Query` class.

* Notice that `relevant_document_ids` represents the *ground-truth labels*, which is used to grade the retriever afterwards.

* The datasets that are used in (main.py) come annotated. Each FiQA question ships with the passages that answer it (ground_truths), and each SQuAD question ships with the context paragraph it was written from.


In [9]:
from src.data_utils.query import Query

class DevQuery(Query):
    def __init__(self, text: str, id: str, relevant_document_ids: list[str], label: str):
        
        self.text = text
        self.id = id
        self.relevant_document_ids = relevant_document_ids
        self.label = label

Checking the dummy dataset to mock some GT labels.

In [10]:
for i in range(3):
    print(raw_docs[0]["document_id"][i], raw_docs[0]["text"][i])

d0 the cat sat on the mat
d1 the dog sat on the log
d2 cats and dogs


Instantiating the `Query` class 4 times (as an array), to get a few different example queries

In [12]:
queries = [
    DevQuery(text='is that a giraffe',  id='q0', relevant_document_ids=[''],        label='q0'),    # no common terms with any document
    DevQuery(text='hello dog',          id='q1', relevant_document_ids=['d1'],      label='q1'),    # "dog" in common with d1
    DevQuery(text='cats and dogs',      id='q2', relevant_document_ids=['d2'],      label='q2'),    # all words in common with d2
    DevQuery(text='a dog and a cat',    id='q3', relevant_document_ids=['d0,d1'],   label='q3'),    # "cat" in common with d0, "dog" in common with d1, none in common with d3
]

q0,q1,q2,q3 = queries

Multiple ways to access properties

In [ ]:
print(q1.text)          # using the object - preferred (more transparent)
print(q1.getQuery())    # using the method

hello dogs
hello dogs


We use the methods like this (here used on full array of queries)

In [ ]:
[tf_idf.GetQueryVocabulary(q.text) for q in queries]


[['a', 'that', 'is', 'giraffe'],
 ['dog', 'hello'],
 ['dogs', 'cats', 'and'],
 ['cat', 'a', 'dog', 'and']]

`GetQueryVocabulary` is used to compute the query term counts in `GetQueryTermCounts`

In [59]:
[tf_idf.GetQueryTermCounts(q.text) for q in queries]

[{'a': 3, 'that': 1, 'is': 1, 'giraffe': 1},
 {'dog': 1, 'hello': 1},
 {'dogs': 1, 'cats': 1, 'and': 1},
 {'cat': 1, 'a': 4, 'dog': 1, 'and': 1}]

Converting the queries into vectors.

* The query term counts (`GetQueryTermCounts`) can now be used to vectorize the queries using `QueryToVector`.

* The vectors have one entry per word in the corpus vocabulary, and each entry is how many times that word appears in the query.
* This can then later be used to compare the queries to the documents.

In [ ]:
print(tf_idf.QueryToVector([q1.text]))
# [[0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]]

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1 stored elements and shape (1, 10)>
  Coords	Values
  (0, 8)	1.0


The 1.0 means the word at column 8 appears once in the query. It's a raw count, not a TF-IDF weight.

`q1` is 'hello dog'

'dog' is in the vocabulary, so its count of 1 goes into that word's column, which happens to be column 8.

In [ ]:
{i: t for t, i in tf_idf.term_to_index.items()}[8]   # -> 'dog'

'dog'

In [64]:
[print(tf_idf.QueryToVector([q.text])) for q in queries]

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 0 stored elements and shape (1, 10)>
<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1 stored elements and shape (1, 10)>
  Coords	Values
  (0, 8)	1.0
<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 3 stored elements and shape (1, 10)>
  Coords	Values
  (0, 4)	1.0
  (0, 6)	1.0
  (0, 7)	1.0
<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 3 stored elements and shape (1, 10)>
  Coords	Values
  (0, 4)	1.0
  (0, 8)	1.0
  (0, 9)	1.0


[None, None, None, None]

Now what is left is to calculate scores between the documents and queries.

* CalculateScores returns a dense NumPy array with one row per query and one column per document. Each entry is how well that document matches that query, and higher means a better match.

* Entry (i, j) is the dot product of query i's count vector with document j's TF-IDF row. That is, for every word the query and document share, it multiplies the word's count in the query by the word's TF-IDF weight in the document, then adds those up.

In [69]:
[tf_idf.CalculateScores([q.getQuery()]) for q in queries]

[array([[0., 0., 0.]], dtype=float32),
 array([[0.        , 0.54930615, 0.        ]], dtype=float32),
 array([[0.      , 0.      , 3.295837]], dtype=float32),
 array([[0.54930615, 0.54930615, 1.0986123 ]], dtype=float32)]

#### BM25


In [28]:
from src.retrievers.bm25 import BM25
from src.retrievers.base_retriever import Retriever

class DevBM25(BM25):
    def __init__(self, documents, k1=1.5, b=0.75):
        Retriever.__init__(self, documents)   
        self.k1 = k1
        self.b = b
        
        self.corpus_vocabulary = self.GetCorpusVocabulary()
        self.idf = self.GetInverseDocumentFrequencies()
        self.bm25_matrix = self.GetDocumentsBM25Vectors()


b_dev = DevBM25(docs)

In [29]:
print("Corpus Vocabulary", b_dev.GetCorpusVocabulary())
print("BM25 IDF Values", b_dev.GetInverseDocumentFrequencies())


doc_lens, avg_len = b_dev.GetDocumentLengths()
print("Document Lengths", {getattr(doc, 'id', getattr(doc, '_id', str(doc))): length for doc, length in doc_lens.items()})
print("Average Document Length", avg_len)


print("BM25 Matrix ", b_dev.bm25_matrix.shape)

Corpus Vocabulary {'mat', 'on', 'sat', 'dog', 'cats', 'dogs', 'the', 'and', 'log', 'cat'}
BM25 IDF Values {'mat': np.float64(0.9808292530117262), 'on': np.float64(0.47000362924573563), 'sat': np.float64(0.47000362924573563), 'the': np.float64(0.47000362924573563), 'cat': np.float64(0.9808292530117262), 'dog': np.float64(0.9808292530117262), 'log': np.float64(0.9808292530117262), 'and': np.float64(0.9808292530117262), 'dogs': np.float64(0.9808292530117262), 'cats': np.float64(0.9808292530117262)}
Document Lengths {'d0': 6, 'd1': 6, 'd2': 3}
Average Document Length 5.0
BM25 Matrix  (3, 10)


Query test

In [30]:

print("Query Vectors")
print([b_dev.QueryToVector([q.getQuery()]).toarray() for q in queries])


for q in queries:
    scores = b_dev.CalculateScores([q.getQuery()])
    print(f"Query '{q.id}' ('{q.getQuery()}')")
    print(scores)

Query Vectors
[array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]], dtype=float32), array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]], dtype=float32), array([[0., 0., 0., 0., 1., 1., 0., 1., 0., 0.]], dtype=float32)]
Query 'q0' ('unrelated dogs')
[[0.        0.        1.1961333]]
Query 'q1' ('hello dogs')
[[0.        0.        1.1961333]]
Query 'q2' ('cats and dogs')
[[0.     0.     3.5884]]


In [ ]:

bm25_created= BM25(docs)

print("BM25 scores:")
query_texts = [q.getQuery() for q in queries]
print(bm25_created.CalculateScores(query_texts))

Scores Oficiais do BM25:
[[0.        0.        1.1961333]
 [0.        0.        1.1961333]
 [0.        0.        3.5884   ]]
